# Heart Disease Prediction — Exploratory Data Analysis

**Dataset** : UCI Heart Disease (Cleveland) — 303 patients, 13 features + 1 cible  
**Objectif** : Prédire si un patient souffre d'une maladie cardiaque (`target`: 0 = Non, 1 = Oui)

| Feature | Description |
|---|---|
| age | Âge du patient |
| sex | Sexe (1=Homme, 0=Femme) |
| cp | Type de douleur thoracique (0-3) |
| trestbps | Pression artérielle au repos (mm Hg) |
| chol | Cholestérol sérique (mg/dl) |
| fbs | Glycémie à jeun > 120 mg/dl (1=Oui) |
| restecg | Résultats ECG au repos (0-2) |
| thalach | Fréquence cardiaque maximale |
| exang | Angine d'effort (1=Oui) |
| oldpeak | Dépression ST à l'effort |
| slope | Pente du segment ST (0-2) |
| ca | Nombre de vaisseaux colorés (0-3) |
| thal | Thalassémie (0-3) |
| **target** | **Maladie cardiaque (0=Non, 1=Oui)** |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Chargement et aperçu du dataset

In [ ]:
df = pd.read_csv('../data/heart_disease.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('=== DTYPES ===')
print(df.dtypes)
print(f'\nDoublons: {df.duplicated().sum()}')

## 2. Statistiques descriptives

In [ ]:
df.describe().T

In [ ]:
df.info()

## 3. Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
print(f'Total valeurs manquantes : {missing.sum()}')

plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis')
plt.title('Heatmap des valeurs manquantes')
plt.tight_layout()
plt.show()

## 4. Distribution de la variable cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['target'].value_counts()
axes[0].pie(counts, labels=['No Disease', 'Disease'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0, 0.05))
axes[0].set_title('Distribution du target')

sns.countplot(x='target', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_xticklabels(['No Disease', 'Disease'])
axes[1].set_title('Comptage des classes')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height()} ({p.get_height()/len(df)*100:.1f}%)',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

plt.suptitle('Variable Cible : Maladie Cardiaque', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/target_imbalance.png', dpi=150)
plt.show()

## 5. Distributions des variables numériques

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--',
                    label=f'Mean: {df[col].mean():.1f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=9)

axes[5].axis('off')
plt.suptitle('Distributions des variables numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/numerical_distributions.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x='target', y=col, data=df,
                palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_xticklabels(['No Disease', 'Disease'])
    axes[i].set_title(f'{col} vs Target')

axes[5].axis('off')
plt.suptitle('Boxplots : Variables numériques par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/numerical_boxplots.png', dpi=150)
plt.show()

## 6. Matrice de corrélation (Pearson)

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Matrice de corrélation (Pearson)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=150)
plt.show()

## 7. Variables catégorielles

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'exang', 'slope']

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

for i, feature in enumerate(cat_cols):
    sns.countplot(x=feature, hue='target', data=df, ax=axes[i], palette='muted')
    axes[i].set_title(f'Heart Disease by {feature}')
    axes[i].legend(title='Disease', labels=['No', 'Yes'])

axes[5].axis('off')
plt.tight_layout()
plt.savefig('../data/categorical_impact.png', dpi=150)
plt.show()

In [ ]:
print('=== Taux de maladie par variable catégorielle ===')
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df.groupby(col)['target'].agg(['mean', 'count']).rename(
        columns={'mean': 'DiseaseRate', 'count': 'Total'}
    ).assign(DiseaseRate=lambda x: (x['DiseaseRate']*100).round(1)).to_string())

## 8. Conclusions et observations clés

1. **Dataset équilibré** : ~54% de patients avec maladie vs ~46% sans — pas besoin de SMOTE majeur, mais on l'applique quand même par robustesse.
2. **Âge** : La maladie touche davantage les patients entre 55-65 ans.
3. **thalach** (fréquence cardiaque max) : Les patients *sans* maladie ont un thalach plus élevé — feature discriminante forte.
4. **oldpeak** : Corrélé négativement avec `target` — dépression ST plus faible = moins de maladie.
5. **cp** (douleur thoracique type 1) : Fortement corrélé à la maladie.
6. **exang** (angine d'effort) : Les patients avec angine ont un taux de maladie bien plus élevé.
7. **sex** : Les hommes ont un taux de maladie plus élevé dans ce dataset.
8. **ca** et **thal** : Features importantes à inclure dans le modèle.

**Pipeline modeling** : StandardScaler + SMOTE + GridSearchCV (RandomForest, XGBoost, Stacking)